# AI-Powered Ticket Processing Pipeline - Phase1: Prompt Testing
**Objective:** Evaluate LLM prompt strategies using Chain of Thought (CoT) and enforce structured JSON outputs using Pydantic on the 50-sample dataset.

## 1. Environment Setup and Configuration
Importing required dependencies and configuring the LLM client and projects paths.

In [1]:
import os
import json
import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

In [2]:
try:
  from google.colab import userdata # pyright: ignore[reportMissingImports] 
  API_KEY = userdata.get("OPENAI_API_KEY")

except:
  load_dotenv()
  API_KEY = os.getenv("OPENAI_API_KEY")

CONFIG = {
    "input_sample_file" : "data/poc_sample_50.csv",
    "output_results_file" : "data/poc_results_50.csv",
    "text_col" : "instruction",
    "model_name" : "gpt-5.6-sol",
    "base_url" : "https://api.gapgpt.app/v1",
    "api_key" : API_KEY
}

In [3]:
#  Initialize LLM client
client = OpenAI(
    api_key = CONFIG["api_key"],
    base_url = CONFIG["base_url"]
)

## 2. Load Sampled Dataset
Loading the strategic 50-sample dataset generated in the previous PoC phase.

In [4]:
df_sample = pd.read_csv(CONFIG["input_sample_file"])

print(f"Loaded dataset shape:{df_sample.shape}")
df_sample.head()

Loaded dataset shape:(39, 7)


,flags,instruction,category,intent,response,text_length,noise_score
0,BLQ,i dont know how i can report an error with reg...,ACCOUNT,registration_problems,We understand that you are unsure about how to...,56,0.000000
1,BILQZ,where do i get an compensation of money,REFUND,get_refund,I've become conscious that you are looking for...,39,0.000000
2,BK,track order,ORDER,track_order,Thank you for reaching out! I see you would li...,11,0.000000
3,BCILMW,I paid {{Currency Symbol}}{{Refund Amount}} fo...,REFUND,get_refund,I acknowledge your frustration and urgency in ...,92,0.108696
4,BL,I want help to see in what cases can I ask to ...,REFUND,check_refund_policy,Unquestionably! I completely understand your n...,57,0.000000


## 3. Define Output Schema (Pydantic)
Defin the expected JSON structure for the LLM output, including Moderation, Classification, Sentiment, and Summarization.

In [5]:
class TicketProcessingResult(BaseModel):
    is_safe: bool = Field(description="True if the ticket contains no harmful, offensive, or unsafe content (Moderation).")
    predicted_category: str = Field(description="The main category of the ticket (e.g., Billing, Technical Support, Account, General Inquiry).")
    predicted_sentiment: str = Field(description="The emotional tone of the ticket: Positive, Neutral, or Negative.")
    summary: str = Field(description="A concise, one-sentence summary of the user's issue.")
    suggested_action: str = Field(description="A brief recommended next step for the support agent.")
    reasoning: str = Field(description="Chain of Thought: Step-by-step reasoning explaining how the category, sentiment, and safety were determined.")


## 4. Prompt Engineering (System & User Prompts)
Designing the Chain of Though (CoT) system prompt to instruct the model.

In [6]:
SYSTEM_PROMPT = """You are an expert AI customer support assistant.
Your task is to analyze customer support tickets and extract key information.

Analyze the ticket step-by-step (Chain of Though):
1. First, check if the content is safe and appropriate (Moderation).
2. Determine the core topic to assign a category (Classification).
3. Evaluate the tone of the text (Sentiment Analysis).
4. Summarize the issue in one sentence (Summarization).
5. Suggest a logical next step to resolve the issue.

You MUST return your response entirely in valid JSON format matching the exact keys provided by the user. Do not include Markdown blocks like   json.
"""

In [7]:
def create_user_prompt(ticket_text:str) -> str:
  schema = TicketProcessingResult.model_json_schema()
  return f"Please analyze the following ticket and return a JSON object matching this schema:\n\n{json.dumps(schema, indent=2)}\n\nTicket Text:\n\"\"\"{ticket_text}\"\"\""


## 5. Single Output Testing
Validating the prompt logic and JSON parsing on a single ticket before batch processing.


In [8]:
# Select a sample ticket
sample_text = df_sample.iloc[0][CONFIG["text_col"]]
print(f"Sample Ticket:\n{sample_text}")

Sample Ticket:
i dont know how i can report an error with registrations


In [9]:
# Call the model
response = client.chat.completions.create(
    model = CONFIG["model_name"],
    messages = [
        {"role" : "system", "content": SYSTEM_PROMPT},
        {"role" : "user", "content" : create_user_prompt(sample_text)}
    ],
    temperature = 0.0,
    response_format = {"type" : "json_object"}
)

In [10]:
# Parse and validate using Pydantic
raw_output = response.choices[0].message.content

try:
    parsed_result = TicketProcessingResult.model_validate_json(raw_output)
    print("Successfully Parsed Output:\n")
    print(parsed_result.model_dump_json(indent=2))
except Exception as e:
    print(f"Parsing Error: {e}")
    print(f"Raw Output was:\n{raw_output}")

Successfully Parsed Output:

{
  "is_safe": true,
  "predicted_category": "Technical Support",
  "predicted_sentiment": "Neutral",
  "summary": "The user needs help reporting an error related to registrations.",
  "suggested_action": "Provide the appropriate error-reporting channel and ask for relevant details, including the error message, steps taken, screenshots, and account or registration context.",
  "reasoning": "The ticket contains no harmful or offensive content. It concerns an error involving registrations, so Technical Support is the best category. The wording expresses uncertainty without strong positive or negative emotion, indicating a Neutral sentiment."
}


## 6. Batch Processing (PoC Execution)
Iterating over the 50 sampled tickets and storing the structured responses.

In [11]:
results_list = []

for index,row in tqdm(df_sample.iterrows(), total= df_sample.shape[0], desc = "Processing Tickets"):
    ticket_text = row[CONFIG["text_col"]]

    try:
        res = client.chat.completions.create(
            model = CONFIG["model_name"],
            messages = [ 
                {"role" : "system", "content" : SYSTEM_PROMPT},
                {"role" : "user", "content" : create_user_prompt(ticket_text)},   
            ],
            temperature=0.0,
            response_format = {"type": "json_object"}
        )

        raw_json = res.choices[0].message.content
        parsed = TicketProcessingResult.model_validate_json(raw_json)

        # Merge original data with LLM output
        combined_row = row.to_dict()
        combined_row.update(parsed.model_dump())
        results_list.append(combined_row) 

    except Exception as e:
        print(f"Error processing row {index} : {e}")

        # Add empty data for failed row to maintain alignment
        combined_row = row.to_dict()
        combined_row.update({k:None for k in TicketProcessingResult.model_fields.keys()})

        results_list.append(combined_row)

df_results = pd.DataFrame(results_list)
print(f"Complete processing.\nResults shape: {df_results.shape}")

Processing Tickets: 100%|██████████| 39/39 [06:37<00:00, 10.19s/it]


Complete processing.
Results shape: (39, 13)


## 7. Export Results
Saving the evaluated dataset for manual review or LLM-as-a-Judge evaluation.

In [12]:
# Ensure directory exists
os.makedirs(os.path.dirname(CONFIG["output_results_file"]),exist_ok = True)

In [13]:
# Export to csv
df_results.to_csv(CONFIG["output_results_file"],index = False)
print(f"Final results successfully saved to: {CONFIG["output_results_file"]}")

df_results.head(2)

Final results successfully saved to: data/poc_results_50.csv


,flags,instruction,category,intent,response,text_length,noise_score,is_safe,predicted_category,predicted_sentiment,summary,suggested_action,reasoning
0,BLQ,i dont know how i can report an error with reg...,ACCOUNT,registration_problems,We understand that you are unsure about how to...,56,0.0,True,Technical Support,Neutral,The user needs guidance on how to report a reg...,Explain how to submit an error report and ask ...,The ticket contains no harmful or offensive co...
1,BILQZ,where do i get an compensation of money,REFUND,get_refund,I've become conscious that you are looking for...,39,0.0,True,Billing,Neutral,The customer is asking how to request or recei...,Ask what issue or transaction the compensation...,The ticket contains no harmful or offensive co...
